[![Abrir en Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/apmontesp/Landslides_-Applied-ML-Course/blob/main/visualizacion_datos/entregables/02_comparativa_visual.ipynb)

# Comparativa Visual — Del Hallazgo al Argumento

**Audiencia:** Público general sin formación técnica  
**Pregunta central:** ¿Qué nos dicen estos modelos sobre cómo detectar deslizamientos en Colombia?

---
Este notebook tiene dos partes:
- **Fase Exploratoria** — las gráficas tal como surgen del análisis de datos
- **Fase Aclaratoria** — la misma información reencuadrada como argumento con contexto colombiano

> *Un modelo más sofisticado no siempre gana — y en Colombia, donde no hay un dataset propio,*  
> *elegir mal el modelo y el protocolo puede ser la diferencia entre una herramienta útil y una que falla.*

In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
import seaborn as sns

IN_COLAB = 'google.colab' in __import__('sys').modules
if IN_COLAB:
    if not os.path.exists('Landslides_-Applied-ML-Course'):
        os.system('git clone https://github.com/apmontesp/Landslides_-Applied-ML-Course.git')
    DATA_DIR = 'Landslides_-Applied-ML-Course/visualizacion_datos/data'
    FIG_DIR  = 'Landslides_-Applied-ML-Course/visualizacion_datos/data/figures'
else:
    DATA_DIR = '../data'
    FIG_DIR  = '../data/figures'

os.makedirs(FIG_DIR, exist_ok=True)

plt.rcParams.update({
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.edgecolor': '#CCCCCC', 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#EEEEEE', 'grid.linewidth': 0.8,
    'font.family': 'sans-serif', 'font.size': 11,
    'xtick.color': '#555555', 'ytick.color': '#555555', 'axes.labelcolor': '#333333',
})

COLORES = {
    'RedEdge': '#DC2626', 'Topo': '#F97316', 'SAR': '#EAB308', 'Optico': '#3B82F6',
    'enfasis': '#DC2626', 'gris': '#9CA3AF', 'gris_claro': '#E5E7EB',
    'clasico': '#6B7280', 'dl': '#C4B5FD',
}

df = pd.read_csv(f'{DATA_DIR}/comparison_table.csv')
for col in ['F1 medio','Std','AUC-ROC','Precisión','Recall','IoU']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

FOLDS_FALLBACK = {
    'LR':  [0.7929, 0.7681, 0.8232, 0.7772, 0.7813],
    'SVM': [0.7526, 0.7731, 0.8100, 0.8325, 0.8189],
    'RF':  [0.8363, 0.8238, 0.8480, 0.8401, 0.8361],
    'ResNet-50':      [0.7762, 0.7708, 0.8154, 0.7636, 0.8065],
    'U-Net ResNet34': [0.6855, 0.7084, 0.7061, 0.6900, 0.6842],
}
try:
    def load_f(fname, key='best_f1', alt='f1_pixel_thr05'):
        with open(f'{DATA_DIR}/folds/{fname}') as f:
            d = json.load(f)
        return [x.get(key, x.get(alt, 0)) for x in d['folds']]
    fold_data = {
        'LR':  load_f('logistic_regression_folds.json'),
        'SVM': load_f('svm_folds.json'),
        'RF':  load_f('random_forest_folds.json'),
        'ResNet-50':      load_f('resnet50_folds.json', key='f1_thr05'),
        'U-Net ResNet34': load_f('unet_folds.json', key='f1_pixel_thr05'),
    }
except FileNotFoundError:
    fold_data = FOLDS_FALLBACK.copy()

print('Setup completo.')

---
# Parte 1 — Fase Exploratoria
Las siguientes gráficas muestran los datos tal como surgen del análisis — sin narrativa, sin énfasis dirigido. Son el punto de partida antes de formular cualquier argumento.

### Exploratoria 1 — F1-Score por modelo

In [ ]:
modelos_e = ['LR', 'SVM', 'RF', 'ResNet-50', 'EfficientNet', 'U-Net']
f1_e = [0.7886, 0.7974, 0.8368, 0.7840, 0.7554, 0.4443]
tipos_e = ['Clásico','Clásico','Clásico','DL','DL','DL']
cols_e = [COLORES['clasico'] if t == 'Clásico' else COLORES['dl'] for t in tipos_e]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(modelos_e, f1_e, color=cols_e, height=0.55, zorder=3)
for bar, val in zip(bars, f1_e):
    ax.text(val + 0.004, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10)
ax.set_xlim(0, 0.93)
ax.set_xlabel('F1-Score')
ax.set_title('F1-Score por Modelo (exploratoria — sin énfasis)', fontsize=12)
ax.spines['left'].set_visible(False)
ax.tick_params(axis='y', length=0)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/comp_e1_f1_exploratorio.png', dpi=150, bbox_inches='tight')
plt.show()

### Exploratoria 2 — Precisión vs Cobertura

In [ ]:
datos_pr_e = {'LR':(0.7971,0.7806), 'SVM':(0.8193,0.7777),
              'RF':(0.7439,0.9569), 'ResNet-50':(0.7219,0.8771)}

fig, ax = plt.subplots(figsize=(6, 5.5))
r_arr = np.linspace(0.01, 0.999, 300)
for f1_val in [0.72, 0.78, 0.84]:
    p_arr = f1_val * r_arr / (2 * r_arr - f1_val)
    mask = (p_arr > 0) & (p_arr <= 1.0)
    ax.plot(r_arr[mask], p_arr[mask], color='#E5E7EB', lw=1.0, ls='--')
    ax.text(r_arr[mask][int(sum(mask)*0.3)], p_arr[mask][int(sum(mask)*0.3)]+0.01,
            f'F1={f1_val}', fontsize=7.5, color='#9CA3AF')
for nm, (prec, rec) in datos_pr_e.items():
    ax.scatter(rec, prec, s=80, color=COLORES['clasico'], zorder=5)
    ax.text(rec+0.005, prec+0.008, nm, fontsize=9)
ax.set_xlim(0.65, 1.01)
ax.set_ylim(0.65, 0.88)
ax.set_xlabel('Cobertura (Recall)')
ax.set_ylabel('Precisión')
ax.set_title('Precisión vs Cobertura (exploratoria)', fontsize=12)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/comp_e2_pr_exploratorio.png', dpi=150, bbox_inches='tight')
plt.show()

### Exploratoria 3 — Variabilidad entre folds

In [ ]:
modelos_f = list(fold_data.keys())
valores_f = list(fold_data.values())

fig, ax = plt.subplots(figsize=(9, 4.5))
bp = ax.boxplot(valores_f, vert=False, patch_artist=True,
                widths=0.45, showfliers=False,
                medianprops=dict(color='white', lw=2),
                whiskerprops=dict(color='#9CA3AF'),
                capprops=dict(color='#9CA3AF'),
                boxprops=dict(linewidth=0))
for patch in bp['boxes']:
    patch.set_facecolor(COLORES['clasico'])
    patch.set_alpha(0.6)
for i, vals in enumerate(valores_f, start=1):
    ax.scatter(vals, [i]*len(vals), color='#374151', s=35, zorder=5, alpha=0.8)
ax.set_yticks(range(1, len(modelos_f)+1))
ax.set_yticklabels(modelos_f)
ax.set_xlabel('F1-Score')
ax.set_title('Variabilidad entre Folds (exploratoria — sin énfasis)', fontsize=12)
ax.spines['left'].set_visible(False)
ax.tick_params(axis='y', length=0)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/comp_e3_folds_exploratorio.png', dpi=150, bbox_inches='tight')
plt.show()

---
# Parte 2 — Fase Aclaratoria
Las mismas preguntas, ahora reencuadradas con un argumento. Cada gráfica tiene un mensaje único, anotaciones que guían la interpretación, y contexto sobre lo que implica para Colombia.

> **Dos mensajes centrales:**
> 1. El resultado cambia según cómo evalúes el modelo — eso es una trampa metodológica
> 2. Colombia tiene alta exposición a deslizamientos pero ningún dataset etiquetado propio — eso define qué modelo sirve

---
### Aclaratoria 1 — La trampa de evaluar mal
**Mensaje:** *El mismo modelo puede verse excelente o mediocre dependiendo de cómo lo evalúes.*

Los modelos clásicos se evaluaron con dos protocolos distintos:
- **14 bandas del satélite** (protocolo optimizado, n=1500 muestras)
- **Características básicas del terreno** (HOG+DEM+NDVI, comparable con la literatura, n=3799)

Si alguien en Colombia implementara un modelo y solo reportara el protocolo más favorable, podría sobrestimar el rendimiento real en condiciones locales.

In [ ]:
# Dos protocolos para modelos clásicos
modelos_a1 = ['LR', 'SVM', 'RF']
f1_opt  = [0.7886, 0.7974, 0.8368]   # 14 bandas, n=1500
f1_base = [0.7512, 0.7340, 0.7891]   # características básicas, n=3799
f1_dl   = {'ResNet-50': 0.7840, 'EfficientNet': 0.7554, 'U-Net': 0.4443}

# Colores: azul oscuro = 14 bandas, azul claro = básicas, gris = DL
COLOR_OPT  = '#1E3A5F'   # azul oscuro
COLOR_BASE = '#93C5FD'   # azul claro
COLOR_DL   = '#D1D5DB'   # gris neutro

x = np.arange(len(modelos_a1))
ancho = 0.32

fig, ax = plt.subplots(figsize=(10, 5.5))

# --- Barras DL ---
dl_nombres = list(f1_dl.keys())
dl_vals    = list(f1_dl.values())
x_dl = np.arange(len(modelos_a1), len(modelos_a1) + len(dl_nombres))
ax.bar(x_dl, dl_vals, width=ancho * 1.9, color=COLOR_DL, zorder=3)
for xi, val in zip(x_dl, dl_vals):
    ax.text(xi, val + 0.008, f'{val:.3f}', ha='center', fontsize=9.5, color='#6B7280')

# --- Barras clásicos ---
b1 = ax.bar(x - ancho/2, f1_opt,  width=ancho, color=COLOR_OPT, zorder=4)
b2 = ax.bar(x + ancho/2, f1_base, width=ancho, color=COLOR_BASE, zorder=4)

for bar, val in zip(b1, f1_opt):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.008, f'{val:.3f}',
            ha='center', fontsize=9.5, color='#1E3A5F', fontweight='bold')
for bar, val in zip(b2, f1_base):
    ax.text(bar.get_x()+bar.get_width()/2, val+0.008, f'{val:.3f}',
            ha='center', fontsize=9.5, color='#374151')

# --- Flecha Δ RF (solo la flecha + Δ, sin texto adicional) ---
rf_idx = 2
ax.annotate('', xy=(rf_idx + ancho/2, f1_base[rf_idx] + 0.004),
            xytext=(rf_idx - ancho/2, f1_opt[rf_idx] + 0.004),
            arrowprops=dict(arrowstyle='<->', color=COLORES['enfasis'], lw=2))
mid_y = (f1_opt[rf_idx] + f1_base[rf_idx]) / 2 + 0.03
ax.text(rf_idx + 0.04, mid_y, f'Δ = {f1_opt[rf_idx]-f1_base[rf_idx]:.3f}',
        fontsize=9, color=COLORES['enfasis'], fontweight='bold')

# --- Línea F1=0.80 ---
ax.axhline(0.80, color='#374151', lw=1.2, ls='--', zorder=2)
ax.text(len(modelos_a1)+len(dl_nombres)-0.3, 0.803, 'F1 = 0.80', fontsize=8.5, color='#374151')

# --- Eje X ---
ax.set_xticks(list(x) + list(x_dl))
ax.set_xticklabels(modelos_a1 + dl_nombres)
ax.set_ylim(0.35, 0.96)
ax.set_xlabel('Modelo')
ax.set_ylabel('F1-Score')

# --- Separador clásico / DL en la parte superior ---
sep_x = len(modelos_a1) - 0.5
ax.axvline(sep_x, color='#E5E7EB', lw=1.5)
ax.text(len(modelos_a1)/2 - 0.2, 0.93, 'Modelos clásicos',
        fontsize=8.5, color='#9CA3AF', ha='center')
ax.text(len(modelos_a1) + len(dl_nombres)/2 - 0.3, 0.93, 'Deep Learning',
        fontsize=8.5, color='#9CA3AF', ha='center')

ax.set_title('El protocolo de evaluación cambia el resultado — mismos modelos, distintas condiciones',
             fontsize=11, pad=12)

# --- Leyenda arriba derecha, sin solapar ---
leyenda = [
    mpatches.Patch(color=COLOR_OPT,  label='14 bandas del satélite (optimizado)'),
    mpatches.Patch(color=COLOR_BASE, label='Características básicas del terreno'),
    mpatches.Patch(color=COLOR_DL,   label='Deep Learning'),
]
ax.legend(handles=leyenda, loc='upper right', fontsize=9, frameon=True,
          framealpha=0.95, edgecolor='#E5E7EB', bbox_to_anchor=(0.99, 0.99))

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/comp_a1_trampa_protocolos.png', dpi=150, bbox_inches='tight')
plt.show()


---
### Aclaratoria 2 — Colombia frente a la literatura global
**Mensaje:** *Los benchmarks globales están entrenados en terrenos distintos al colombiano.*

Colombia registra entre 400 y 600 eventos de deslizamiento por año (SGC, 2023 — uno de los países con mayor frecuencia en América Latina). Sin embargo, no existe un dataset etiquetado colombiano. Los modelos de referencia internacional se entrenaron en Nepal, Perú e Italia — con regímenes de lluvia, pendientes y cobertura vegetal distintos a los Andes colombianos.

¿Dónde caen nuestros modelos frente a esos benchmarks?

In [ ]:
datos_pr_a2 = {
    'LR':        (0.7971, 0.7806),
    'SVM':       (0.8193, 0.7777),
    'RF':        (0.7439, 0.9569),
    'ResNet-50': (0.7219, 0.8771),
}
literatura_a2 = {
    'Ghorbanzadeh 2022 (Nepal)':        0.717,
    'L4S Competition 2022 (multi-reg)': 0.739,
    'Multi-scale CNN 2024 (Italia)':    0.760,
    'Enhanced U-Net++ 2025 (Perú)':     0.841,
}

fig, ax = plt.subplots(figsize=(8, 6.5))
r_arr = np.linspace(0.01, 0.999, 400)

# Curvas ISO-F1 — degradado rojo
colores_lit = ['#FECACA', '#F87171', '#EF4444', '#B91C1C']
# posición x fija para las etiquetas (aprox Recall=0.68)
label_r = 0.68
for (nombre, f1_val), cl in zip(literatura_a2.items(), colores_lit):
    p_arr = f1_val * r_arr / (2 * r_arr - f1_val)
    mask = (p_arr > 0) & (p_arr <= 1.0)
    ax.plot(r_arr[mask], p_arr[mask], color=cl, lw=1.3, ls='--', zorder=2)
    # etiqueta alineada a la misma columna x
    p_label = f1_val * label_r / (2 * label_r - f1_val)
    if 0 < p_label <= 1.0:
        ax.text(label_r - 0.012, p_label + 0.009, nombre,
                fontsize=7.5, color=cl, ha='left')

# Puntos modelos
col_mod  = {'LR':'#6B7280','SVM':'#6B7280','RF':COLORES['enfasis'],'ResNet-50':'#374151'}
tipo_mod = {'LR':'Clásico','SVM':'Clásico','RF':'Clásico','ResNet-50':'Deep Learning'}
marker_mod = {'LR':'o','SVM':'o','RF':'o','ResNet-50':'D'}
for nm, (prec, rec) in datos_pr_a2.items():
    ax.scatter(rec, prec, s=130, color=col_mod[nm], zorder=6,
               edgecolors='white', linewidth=1.5, marker=marker_mod[nm])
    offsets = {'LR':(-0.016, 0.013),'SVM':(0.006, 0.013),
               'RF':(0.006, -0.024),'ResNet-50':(0.006, 0.013)}
    dx, dy = offsets[nm]
    ax.text(rec+dx, prec+dy, nm, fontsize=9.5,
            color=col_mod[nm], fontweight='bold')

# Zona Colombia — esquina inferior izquierda, sin cruzar puntos
ax.axhspan(0.60, 0.66, alpha=0.07, color='#FCD34D', zorder=1)
ax.text(0.635, 0.607, 'Sin datos colombianos — zona de incertidumbre',
        fontsize=7.5, color='#92400E', style='italic')

# Anotación RF — reubicada abajo derecha
ax.annotate('RF: mayor cobertura,\nalcanza benchmark 2025',
            xy=(0.9569, 0.7439), xytext=(0.83, 0.655),
            arrowprops=dict(arrowstyle='->', color=COLORES['enfasis'], lw=1.5),
            fontsize=8.5, color=COLORES['enfasis'])

ax.set_xlim(0.62, 1.02)
ax.set_ylim(0.58, 0.89)
ax.set_xlabel('Cobertura (Recall) — fracción de deslizamientos reales detectados')
ax.set_ylabel('Precisión — de las alertas, ¿cuántas son reales?')
ax.set_title('Nuestros modelos vs. Benchmarks globales\n(curvas punteadas = papers internacionales, escala F1)',
             fontsize=11, pad=10)

# Leyenda con distinción clásico / DL
leyenda_a2 = [
    Line2D([0],[0], marker='o', color='w', markerfacecolor='#6B7280',
           markersize=10, label='Modelos clásicos'),
    Line2D([0],[0], marker='D', color='w', markerfacecolor='#374151',
           markersize=9, label='Deep Learning'),
    Line2D([0],[0], marker='o', color='w', markerfacecolor=COLORES['enfasis'],
           markersize=10, label='RF — mejor F1 global'),
    Line2D([0],[0], color='#EF4444', ls='--', lw=1.5,
           label='Benchmarks literatura (ISO-F1)'),
]
ax.legend(handles=leyenda_a2, loc='upper left', fontsize=8.5,
          frameon=True, framealpha=0.95, edgecolor='#E5E7EB')

plt.tight_layout()
plt.savefig(f'{FIG_DIR}/comp_a2_colombia_vs_literatura.png', dpi=150, bbox_inches='tight')
plt.show()


---
### Aclaratoria 3 — Más complejo no es mejor
**Mensaje:** *U-Net es la arquitectura más sofisticada — y tiene el peor resultado.*

La objeción más común al ver estos resultados es: *'debe ser un error, las redes profundas siempre son mejores'.*  
Los datos muestran lo contrario, y hay una razón: la arquitectura correcta depende del contexto regional.

Para Colombia, tres factores determinan qué modelo es apropiado:
1. **Disponibilidad de datos etiquetados** — U-Net necesita miles de imágenes segmentadas; RF funciona con cientos
2. **Bandas satelitales disponibles** — RedEdge y SAR son las más discriminativas, pero no siempre accesibles
3. **Interpretabilidad** — un organismo de gestión de riesgos necesita saber *por qué* el modelo alerta, no solo que lo hace

In [ ]:
modelos_a3 = ['U-Net\nResNet34', 'EfficientNet', 'ResNet-50', 'LR', 'SVM', 'RF']
f1_a3      = [0.4443, 0.7554, 0.7840, 0.7886, 0.7974, 0.8368]
complejidad = ['Alta','Alta','Alta','Baja','Media','Media']
col_a3 = [
    '#C4B5FD','#C4B5FD','#C4B5FD',   # DL — violeta claro
    '#9CA3AF','#9CA3AF',              # clásicos — gris
    COLORES['enfasis'],               # RF — rojo
]

fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.barh(modelos_a3, f1_a3, color=col_a3, height=0.55, zorder=3)

# Etiqueta de complejidad dentro de la barra
for bar, comp in zip(bars, complejidad):
    bw = bar.get_width()
    ax.text(min(bw * 0.05, 0.04), bar.get_y() + bar.get_height()/2,
            f'Complejidad: {comp}', va='center', fontsize=8.5,
            color='white', fontweight='bold')

# Valor al final de la barra
for bar, val in zip(bars, f1_a3):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10, color='#374151')

# Línea F1=0.80
ax.axvline(0.80, color='#374151', lw=1.2, ls='--', zorder=2)
ax.text(0.801, 5.65, 'F1 = 0.80', fontsize=8.5, color='#374151')

# Etiqueta mínima en RF
ax.text(0.842, 5, 'apto para Colombia', fontsize=8.5,
        color=COLORES['enfasis'], va='center')

ax.set_xlim(0, 0.97)
ax.set_xlabel('F1-Score — capacidad de detección')
ax.set_ylabel('Modelo')
ax.set_title('Complejidad del modelo vs. resultado real\nMás parámetros no garantizan mejor detección',
             fontsize=11, pad=12)

# Leyenda
leyenda_a3 = [
    mpatches.Patch(color='#C4B5FD', label='Deep Learning'),
    mpatches.Patch(color='#9CA3AF', label='Modelos clásicos'),
    mpatches.Patch(color=COLORES['enfasis'], label='Mejor resultado'),
]
ax.legend(handles=leyenda_a3, loc='lower right', fontsize=9, frameon=False)
ax.spines['left'].set_visible(False)
ax.tick_params(axis='y', length=0)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/comp_a3_complejidad_vs_f1.png', dpi=150, bbox_inches='tight')
plt.show()


---
### Aclaratoria 4 — ¿Qué información satelital necesita Colombia?
**Mensaje:** *Las bandas que más discriminan son RedEdge — disponibles en Sentinel-2, con cobertura nacional.*

La buena noticia: Sentinel-2 cubre Colombia con revisita de 5 días y las bandas RedEdge (B5, B6, B7) son gratuitas y accesibles desde la API de Copernicus. El desafío no es el satélite — es tener imágenes etiquetadas post-evento para entrenar.

In [ ]:
# Ordenado de mayor a menor Δ (el más discriminativo arriba)
canales_a4 = [
    ('S2-B7 RedEdge3', 0.8073, 'RedEdge', True),
    ('S2-B6 RedEdge2', 0.5625, 'RedEdge', True),
    ('ALOS DEM',       0.1954, 'Topo',    False),
    ('S1-VH SAR',      0.1882, 'SAR',     True),
    ('DEM Slope',      0.0430, 'Topo',    False),
    ('S2-B8A NIR-A',   0.0221, 'Optico',  True),
]

nombres_a4 = [c[0] for c in canales_a4]
deltas_a4  = [c[1] for c in canales_a4]
grupos_a4  = [c[2] for c in canales_a4]
disponible = [c[3] for c in canales_a4]
cols_a4    = [COLORES[g] for g in grupos_a4]

fig, ax = plt.subplots(figsize=(9.5, 4.2))
bars = ax.barh(nombres_a4, deltas_a4, color=cols_a4, height=0.55, zorder=3)

# Valor al final de la barra
for bar, val in zip(bars, deltas_a4):
    ax.text(val + 0.012, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=10, color='#374151')

# Disponibilidad: texto pequeño en gris / verde, pegado al eje
for bar, disp in zip(bars, disponible):
    estado = 'Copernicus' if disp else 'DEM externo'
    color_est = '#16A34A' if disp else '#9CA3AF'
    ax.text(0.005, bar.get_y() + bar.get_height()/2,
            estado, va='center', fontsize=7.5,
            color=color_est, alpha=0.85)

ax.set_xlim(0, 0.95)
ax.set_xlabel('Brecha de señal (Δ) entre zonas con y sin deslizamiento')
ax.set_ylabel('Canal satelital')
ax.set_title('Canales más discriminativos y su disponibilidad para Colombia',
             fontsize=11, pad=10)

# Leyenda arriba derecha
leyenda_a4 = [
    mpatches.Patch(color=COLORES['RedEdge'], label='Banda RedEdge (Sentinel-2)'),
    mpatches.Patch(color=COLORES['Topo'],    label='Topografía (DEM/Pendiente)'),
    mpatches.Patch(color=COLORES['SAR'],     label='Radar SAR (Sentinel-1)'),
    mpatches.Patch(color=COLORES['Optico'],  label='Óptico NIR (Sentinel-2)'),
]
ax.legend(handles=leyenda_a4, loc='upper right', fontsize=8.5,
          frameon=True, framealpha=0.95, edgecolor='#E5E7EB')

ax.spines['left'].set_visible(False)
ax.tick_params(axis='y', length=0)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/comp_a4_canales_colombia.png', dpi=150, bbox_inches='tight')
plt.show()


---
### Aclaratoria 5 — ¿Cuál modelo confiar en condiciones no vistas?
**Mensaje:** *La consistencia entre experimentos es la señal de confianza — no la media más alta.*

Un modelo que varía mucho entre folds puede haber 'memorizado' el conjunto de entrenamiento. En Colombia, donde los datos serían escasos y de un solo evento, la variabilidad entre folds es un indicador crítico de qué tan bien funcionaría el modelo en un deslizamiento nuevo y no visto.

In [ ]:
modelos_a5 = list(fold_data.keys())
valores_a5 = list(fold_data.values())
medias_a5  = [np.mean(v) for v in valores_a5]
stds_a5    = [np.std(v, ddof=1) for v in valores_a5]

orden_a5    = np.argsort(medias_a5)[::-1]
modelos_ord = [modelos_a5[i] for i in orden_a5]
valores_ord = [valores_a5[i] for i in orden_a5]
medias_ord  = [medias_a5[i] for i in orden_a5]
stds_ord    = [stds_a5[i] for i in orden_a5]

col_a5 = [COLORES['enfasis'] if 'RF' in m else '#9CA3AF' for m in modelos_ord]

fig, ax = plt.subplots(figsize=(9, 5))
bp = ax.boxplot(valores_ord, vert=False, patch_artist=True,
                widths=0.45, showfliers=False,
                medianprops=dict(color='white', lw=2),
                whiskerprops=dict(color='#9CA3AF'),
                capprops=dict(color='#9CA3AF'),
                boxprops=dict(linewidth=0))
for patch, col in zip(bp['boxes'], col_a5):
    patch.set_facecolor(col)
    patch.set_alpha(0.75)

rng = np.random.default_rng(42)
for i, (vals, col) in enumerate(zip(valores_ord, col_a5), start=1):
    jitter = rng.uniform(-0.12, 0.12, len(vals))
    ax.scatter(vals, [i + j for j in jitter], color=col, s=50, zorder=5, alpha=0.9)

# Media + std al costado derecho
for i, (med, std, col) in enumerate(zip(medias_ord, stds_ord, col_a5), start=1):
    ax.text(med + 0.003, i + 0.28,
            f'x̄={med:.3f}  Std={std:.3f}',
            fontsize=8.5, color=col, va='bottom')

# Línea referencia
ax.axvline(0.80, color='#374151', lw=1.2, ls='--', zorder=2)
ax.text(0.801, 0.55, 'F1 = 0.80', fontsize=8.5, color='#374151')

# Anotación RF — reubicada arriba derecha, sin cruzar datos
rf_pos = modelos_ord.index('RF') + 1
ax.annotate('Dispersión más pequeña\n→ más confiable en datos nuevos',
            xy=(np.mean(valores_ord[modelos_ord.index('RF')]), rf_pos),
            xytext=(0.700, rf_pos + 1.5),
            arrowprops=dict(arrowstyle='->', color=COLORES['enfasis'], lw=1.5),
            fontsize=8.5, color=COLORES['enfasis'],
            bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                      edgecolor=COLORES['enfasis'], alpha=0.9))

ax.set_yticks(range(1, len(modelos_ord)+1))
ax.set_yticklabels(modelos_ord)
ax.set_xlabel('F1-Score por fold — cada punto es un experimento independiente')
ax.set_ylabel('Modelo')
ax.set_title('Consistencia del modelo como indicador de confianza', fontsize=13, pad=10)
ax.spines['left'].set_visible(False)
ax.tick_params(axis='y', length=0)
plt.tight_layout()
plt.savefig(f'{FIG_DIR}/comp_a5_consistencia_colombia.png', dpi=150, bbox_inches='tight')
plt.show()


---
## Conclusión — Lo que necesita Colombia

Los datos apuntan a tres condiciones para implementar detección de deslizamientos en Colombia:

| Condición | Estado actual | Implicación |
|-----------|--------------|-------------|
| **Dataset etiquetado nacional** | No existe | Sin esto, cualquier modelo es una extrapolación |
| **Bandas satelitales disponibles** | Sentinel-2 disponible (RedEdge incluido) | La señal está — faltan etiquetas |
| **Protocolo de evaluación honesto** | Depende del estudio | Usar 5 folds con protocolo comparable a literatura |

> **La arquitectura viene después del contexto.**  
> Random Forest, interpretable y eficiente con pocos datos, es un punto de partida más sólido para Colombia  
> que redes profundas diseñadas para datasets de miles de imágenes segmentadas.

---
*Datos: Landslide4Sense Dataset · Modelos entrenados en Google Colab · Benchmarks: Ghorbanzadeh (2022), L4S Competition (2022), Multi-scale CNN (2024), Enhanced U-Net++ (2025)*